# Part 3 - 01 Data discovery

NeuroVision AI Part 3: from classification to **detection** (YOLOv8).
Dataset: *MRI for Brain Tumor with Bounding Boxes* (Kaggle), extracted in `data/raw_bbox`.

Goals:
1. Inspect the folder structure and the file types
2. Read an annotation: `class x_center y_center width height`, normalized between 0 and 1
3. Display images with boxes and class names (coordinates converted back to pixels)
4. Confirm the mapping between class folders and class ids
5. Look at the class distribution, images with several boxes, box sizes and image sizes

Nothing is modified here: this notebook only reads the data.

In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from neurovision.yolo_data import (CLASS_NAMES, SEED, build_inventory, draw_boxes,
                                   image_size, set_seed)

set_seed(SEED)

ROOT = Path("../data/raw_bbox")
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)
print("Dataset found:", ROOT.exists())

## 1. Folder structure and file types

In [ ]:
for d in sorted(p for p in ROOT.rglob("*") if p.is_dir() and len(p.relative_to(ROOT).parts) <= 2):
    print(d.relative_to(ROOT))

print()
print("Files per extension:", dict(Counter(p.suffix.lower() for p in ROOT.rglob("*") if p.is_file())))

## 2. One annotation

In [ ]:
example = next(p for p in sorted(ROOT.rglob("*.txt")) if p.stat().st_size > 0)
print(example.relative_to(ROOT))
print(example.read_text())

Each line is `class x_center y_center width height`, all normalized between 0 and 1.
To draw a box in pixels: multiply `x_center` and `width` by the image width, `y_center` and `height` by the image height,
then `x1 = xc - w/2` and `y1 = yc - h/2`.

## 3. Inventory (lightweight: no hashing, no readability check yet)

In [ ]:
df = build_inventory(ROOT, check_readable=False, compute_hash=False)
df["folder_name"] = df["folder_class"].map(lambda i: CLASS_NAMES[int(i)] if pd.notna(i) else "unknown")

print("Files (without .txt/.yaml):", len(df))
print("Label status of images:", df.loc[df.ext_ok, "label_status"].value_counts().to_dict())
pd.crosstab(df["folder_name"], df["split"], margins=True)

## 4. Images with their boxes (3 per class)

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(11, 14))
for cid, row_axes in enumerate(axes):
    pool = df[(df.folder_class == cid) & (df.label_status == "ok")]
    for ax, (_, r) in zip(row_axes, pool.sample(min(3, len(pool)), random_state=SEED).iterrows()):
        ax.imshow(draw_boxes(r.path, r.label_path))
        ax.set_title(f"{CLASS_NAMES[cid]} | {Path(r.rel).name}", fontsize=8)
    for ax in row_axes:
        ax.axis("off")
plt.tight_layout()
plt.savefig(REPORTS / "part3_samples_with_boxes.png", dpi=150)
plt.show()

## 5. Class folders versus class ids found in the labels

In [ ]:
ok = df[df.label_status == "ok"]
box_df = ok.assign(box_class=ok["classes"]).explode("box_class", ignore_index=True)
box_df["box_name"] = box_df["box_class"].map(lambda i: CLASS_NAMES[int(i)])

mapping = pd.crosstab(box_df["folder_name"], box_df["box_name"])
mapping

In [ ]:
# The main class of each folder must be the id we assume (Glioma=0, Meningioma=1, No Tumor=2, Pituitary=3)
for cid, name in enumerate(CLASS_NAMES):
    main = box_df.loc[box_df.folder_name == name, "box_class"].mode().iloc[0]
    print(f"{name:11s} folder -> most frequent box class id {main}  (expected {cid})", "OK" if main == cid else "MISMATCH")

## 6. Class distribution: images and boxes

In [ ]:
images_per_class = pd.crosstab(df.loc[df.ext_ok, "folder_name"], df.loc[df.ext_ok, "split"])
boxes_per_class = pd.crosstab(box_df["box_name"], box_df["split"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
images_per_class.plot.bar(ax=axes[0], title="Images per class folder", rot=0)
boxes_per_class.plot.bar(ax=axes[1], title="Boxes per class id", rot=0)
plt.tight_layout()
plt.savefig(REPORTS / "part3_class_distribution.png", dpi=150)
plt.show()

totals = box_df["box_name"].value_counts()
print(totals.to_dict())
print(f"Imbalance (largest / smallest class, boxes): {totals.max() / totals.min():.2f}")

## 7. Images with several boxes or boxes of another class

In [ ]:
print("Boxes per image:", ok["n_boxes"].value_counts().sort_index().to_dict())

is_mixed = ok.apply(lambda r: set(r.classes) != {int(r.folder_class)} if pd.notna(r.folder_class) else False, axis=1)
mixed = ok[is_mixed]
print(f"\nImages with a box of a class different from their folder: {len(mixed)}")
print(mixed.groupby("folder_name").size().to_dict())

In [ ]:
# Look at them before deciding anything: real multi-tumor images, or labeling errors?
if len(mixed):
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (_, r) in zip(axes.ravel(), mixed.sample(min(6, len(mixed)), random_state=SEED).iterrows()):
        ax.imshow(draw_boxes(r.path, r.label_path))
        ax.set_title(f"{r.folder_name} folder, ids={sorted(set(r.classes))}", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. What a No Tumor annotation looks like, and box sizes

In [ ]:
nt = df[(df.folder_class == 2) & (df.label_status == "ok")]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, r) in zip(axes, nt.sample(min(3, len(nt)), random_state=SEED).iterrows()):
    ax.imshow(draw_boxes(r.path, r.label_path))
    ax.axis("off")
plt.show()

sizes = pd.DataFrame([{"class": CLASS_NAMES[c], "width": w, "height": h, "area": w * h}
                      for r in ok.itertuples() for c, _, _, w, h in r.boxes])
sizes.groupby("class")[["width", "height", "area"]].agg(["mean", "min", "max"]).round(3)

## 9. Image sizes (to choose `imgsz` later)

In [ ]:
shape_counts = Counter(image_size(p) for p in df.loc[df.ext_ok, "path"])
print(len(shape_counts), "distinct sizes (width, height)")
print(shape_counts.most_common(5))

## Findings (fill in with your own numbers)

- **Structure:** Train / Valid (/ Test?) > one folder per class, image and `.txt` side by side? 
- **Annotation format:** YOLO `class xc yc w h`, normalized.
- **Class ids:** 0 = Glioma, 1 = Meningioma, 2 = No Tumor, 3 = Pituitary (confirmed by the folder/id table).
- **"No Tumor":** annotated with a box of class 2, not an empty file.
- **Imbalance:** class 2 (No Tumor) has the fewest boxes.
- **Mixed images:** number of images with boxes of another class, and what the examples show.
- **Image sizes:** most common size(s).

Next: `part3_02_data_preprocessing.ipynb` (cleaning, split, `data.yaml`).